# CardioIA - Análise Inteligente de Sinais Cardíacos
## Monitoramento, detecção de anomalias e visualizações

In [ ]:
print("CardioIA - Monitoramento Cardíaco IoT")
print("Análise de dados, detecção de anomalias e visualizações")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Bibliotecas carregadas!")

In [ ]:
np.random.seed(42)
n_amostras = 100
base_ts = int(pd.Timestamp.now().timestamp() * 1000)
registros = []

for i in range(n_amostras):
    bpm = int(np.random.normal(78, 15))
    if np.random.random() < 0.1:
        bpm = int(np.random.uniform(122, 145))
    temp = round(float(np.random.normal(36.4, 0.7)), 1)
    if np.random.random() < 0.06:
        temp = round(float(np.random.uniform(38.1, 39.8)), 1)
    registros.append({
        'timestamp': base_ts - i * 5000,
        'paciente_id': 'PAC-0001',
        'temperatura': temp,
        'umidade': int(np.random.uniform(50, 85)),
        'bpm': max(40, min(190, bpm)),
        'conectado': bool(np.random.choice([True, True, True, False]))
    })

df = pd.DataFrame(registros)
df['data_hora'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df.sort_values('data_hora')
print(f"✅ Gerados {len(df)} registros simulados")
df.head(10)

In [ ]:
print("=" * 50)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 50)
display(df[['bpm', 'temperatura', 'umidade']].describe())
print(f"\n📍 Valores nulos: {df.isnull().sum().sum()}")
print(f"📍 Conectado: {df['conectado'].mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0,0].hist(df['bpm'], bins=15, color='#dc3545', edgecolor='white')
axes[0,0].set_title('Distribuição de BPM')
axes[0,0].axvline(120, color='red', ls='--', label='Limiar alerta (120)')
axes[0,0].legend()

axes[0,1].hist(df['temperatura'], bins=15, color='#0d6efd', edgecolor='white')
axes[0,1].set_title('Distribuição de Temperatura')
axes[0,1].axvline(38, color='red', ls='--', label='Limiar alerta (38°C)')
axes[0,1].legend()

axes[0,2].hist(df['umidade'], bins=15, color='#198754', edgecolor='white')
axes[0,2].set_title('Distribuição de Umidade')

axes[1,0].boxplot(df['bpm'])
axes[1,0].set_title('Boxplot - BPM')
axes[1,0].axhline(120, color='red', ls='--')

axes[1,1].boxplot(df['temperatura'])
axes[1,1].set_title('Boxplot - Temperatura')
axes[1,1].axhline(38, color='red', ls='--')

axes[1,2].boxplot(df['umidade'])
axes[1,2].set_title('Boxplot - Umidade')

plt.tight_layout()
plt.show()

In [ ]:
df['zscore_bpm'] = (df['bpm'] - df['bpm'].mean()) / df['bpm'].std()
df['anomalia_bpm'] = abs(df['zscore_bpm']) > 2

q1_temp, q3_temp = df['temperatura'].quantile([0.25, 0.75])
iqr_temp = q3_temp - q1_temp
df['anomalia_temp'] = (df['temperatura'] < q1_temp - 1.5*iqr_temp) | (df['temperatura'] > q3_temp + 1.5*iqr_temp)
df['anomalia'] = df['anomalia_bpm'] | df['anomalia_temp']

print(f"Total de anomalias: {df['anomalia'].sum()} de {len(df)} ({df['anomalia'].sum()/len(df)*100:.1f}%)")
print(f"  - BPM anômalo (Z-Score): {df['anomalia_bpm'].sum()}")
print(f"  - Temperatura anômala (IQR): {df['anomalia_temp'].sum()}")

alertas_bpm = (df['bpm'] > 120).sum()
alertas_temp = (df['temperatura'] > 38.0).sum()
print(f"\n📢 Alertas: BPM>120 = {alertas_bpm} | Temp>38°C = {alertas_temp}")
anomalias = df[df['anomalia']][['data_hora','bpm','temperatura','anomalia_bpm','anomalia_temp']]
display(anomalias)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

cores_bpm = ['#dc3545' if a else '#0d6efd' for a in df['anomalia_bpm']]
axes[0,0].scatter(df['data_hora'], df['bpm'], c=cores_bpm, alpha=0.7, s=40)
axes[0,0].axhline(120, color='red', ls='--', label='Alerta 120')
axes[0,0].set_title('BPM ao Longo do Tempo')
axes[0,0].tick_params(axis='x', rotation=45)

cores_temp = ['#dc3545' if a else '#0d6efd' for a in df['anomalia_temp']]
axes[0,1].scatter(df['data_hora'], df['temperatura'], c=cores_temp, alpha=0.7, s=40)
axes[0,1].axhline(38, color='red', ls='--', label='Alerta 38°C')
axes[0,1].set_title('Temperatura ao Longo do Tempo')
axes[0,1].tick_params(axis='x', rotation=45)

axes[0,2].scatter(df['bpm'], df['temperatura'],
                  c=['#dc3545' if a else '#0d6efd' for a in df['anomalia']], alpha=0.6, s=40)
axes[0,2].set_xlabel('BPM')
axes[0,2].set_ylabel('Temperatura (°C)')
axes[0,2].set_title('BPM x Temperatura')

corr = df[['bpm','temperatura','umidade']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,0], vmin=-1, vmax=1, center=0)
axes[1,0].set_title('Matriz de Correlação')

ambos = ((df['bpm']>120) & (df['temperatura']>38)).sum()
s_bpm = ((df['bpm']>120) & ~(df['temperatura']>38)).sum()
s_temp = (~(df['bpm']>120) & (df['temperatura']>38)).sum()
normal = len(df) - ambos - s_bpm - s_temp
axes[1,1].pie([ambos, s_bpm, s_temp, normal],
              labels=['Ambos','Só BPM','Só Temp','Normal'],
              colors=['#dc3545','#ffc107','#fd7e14','#198754'], autopct='%1.1f%%', startangle=90)
axes[1,1].set_title('Proporção de Alertas')

conn = df['conectado'].value_counts()
axes[1,2].bar(['Desconectado','Conectado'], [conn.get(False,0), conn.get(True,0)],
              color=['#6c757d','#0d6efd'], width=0.5)
axes[1,2].set_title('Conectividade')

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 55)
print("   CARDIOIA - RELATÓRIO DE ANÁLISE DE SINAIS")
print("=" * 55)
print(f"\n📅 Período: {df['data_hora'].min().strftime('%d/%m/%Y %H:%M')} até "
      f"{df['data_hora'].max().strftime('%d/%m/%Y %H:%M')}")
print(f"📊 Registros: {len(df)} | Paciente: {df['paciente_id'].iloc[0]}")
print(f"\nBPM médio: {df['bpm'].mean():.1f} | Máx: {df['bpm'].max()} | Mín: {df['bpm'].min()}")
print(f"Temp média: {df['temperatura'].mean():.1f}°C | Máx: {df['temperatura'].max()}°C | Mín: {df['temperatura'].min()}°C")
print(f"\nAlertas BPM>120: {alertas_bpm} ({alertas_bpm/len(df)*100:.1f}%)")
print(f"Alertas Temp>38°C: {alertas_temp} ({alertas_temp/len(df)*100:.1f}%)")
print(f"Anomalias: {df['anomalia'].sum()} ({df['anomalia'].sum()/len(df)*100:.1f}%)")
print(f"Conectado: {df['conectado'].mean()*100:.1f}%")
corr_bpm_temp = df['bpm'].corr(df['temperatura'])
print(f"\nCorrelação BPM-Temperatura: {corr_bpm_temp:.2f}")
if corr_bpm_temp > 0.5:
    print("  → Existe correlação moderada a forte")
print(f"\n{'═' * 55}")
print("   Fim do Relatório")
print(f"{'═' * 55}")

In [ ]:
print("""
   CONCLUSÃO - CardioIA IoT

O sistema demonstrou:
✅ Coleta de sinais vitais (BPM, temperatura, umidade)
✅ Detecção de anomalias (Z-Score e IQR)
✅ Alertas clínicos (BPM>120, Temperatura>38°C)
✅ Edge buffer para resiliência offline

🔗 Projeto Wokwi: https://wokwi.com/projects/464379345574897665
""")